In [8]:
import * as tslab from "tslab";
import { readFileSync } from "fs";

const css = readFileSync("../style.css", "utf-8");
tslab.display.html(`<style>${css}</style>`);

# PicoSat

In TypeScript/JavaScript, there is no direct equivalent to pycosat, but a great alternative is the theorem prover [Z3](https://github.com/Z3Prover/z3) with the npm package `z3-solver`.

In [9]:
import { init } from "z3-solver";

In the Z3 solver, propositional variables are represented using boolean constants:

- A propositional variable \(p\) is created as a boolean variable: `p = Bool("p")`.
- Negation \(\neg p\) is represented with `Not(p)`.
- A clause is represented as a disjunction of literals, for example: `Or(p, Not(q), r)`.
- A CNF formula is a conjunction of such clauses

In [41]:
import { init } from "z3-solver";

async function runExample() {
  const Z3 = await init();
  const ctx = Z3.Context("");

  // Declare boolean variables corresponding to 1, 2, 3
  const p = ctx.Bool.const("p");
  const q = ctx.Bool.const("q");
  const r = ctx.Bool.const("r");

  // Translate each clause from f = [ [1, -2, 3], [-1, 2, -3], [-1, -2, 3], [1, 2, -3] ]
  const clause1 = ctx.Or(p, ctx.Not(q), r);
  const clause2 = ctx.Or(ctx.Not(p), q, ctx.Not(r));
  const clause3 = ctx.Or(ctx.Not(p), ctx.Not(q), r);
  const clause4 = ctx.Or(p, q, ctx.Not(r));

  // Form the CNF conjunction of clauses
  const formula = ctx.And(clause1, clause2, clause3, clause4);

  // Create a solver instance and add the formula
  const solver = new ctx.Solver();
  solver.add(formula);

  // Check satisfiability of the formula
  const result = await solver.check();

  console.log(result.toString());
    if (result.toString() === "sat") {
    const model = solver.model();
    console.log("Model:");
    console.log(model.toString());
  }
}

await runExample();

sat
Model:
(define-fun p () Bool
  false)
(define-fun q () Bool
  false)
(define-fun r () Bool
  false)


This shows that the formula `f` is satisfiable and that the  propositional valuation 
$$ \mathcal{I} = \{ p \mapsto \texttt{False}, q \mapsto \texttt{False}, r \mapsto \texttt{False} \} $$
is a solution for `f`, i.e. we have
$$ \mathcal{I}(\texttt{f}) = \texttt{True}. $$

## Transforming Clauses into PyCoSat Format

In order to use *PicoSat* for the examples discussed in our lecture, we need a function that transforms a formula that is in conjunctive normal form
into the format of *PicoSat*.  Furthermore, we need a function that can translate a solution found by *PicoSat* back into our format.

The function `findVariables` takes a set of `Clauses` and returns the set of all propositional variables
occurring in this set.  

In [11]:
type Variable = string;
type Literal = Variable | ['¬', Variable];
type Clause = Set<Literal>;

In [12]:
function findVariables(Clauses: Clause[]): Set<Variable> {
  const Variables = new Set<Variable>();
  for (const Clause of Clauses) {
    for (const literal of Clause) {
      if (Array.isArray(literal) && literal[0] === '¬') {
        Variables.add(literal[1]);
      } else if (typeof literal === 'string') {
        Variables.add(literal);
      }
    }
  }
  return Variables;
}

The function `numberVariables(Clauses)` takes a set of `Clauses` as input.  It returns two dictionaries:
* The dictionary `Var2Int` maps every propositional variable occurring in `Clauses` to a unique natural number.
* The dictionary `Int2Var` is the mapping that is inverse to the dictionary `Var2Int`.

In [13]:
function numberVariables(Clauses: Clause[]): [Map<Variable, number>, Map<number, Variable>] {
  const Variables = findVariables(Clauses);
  let count = 1;
  const Var2Int = new Map<Variable, number>();
  const Int2Var = new Map<number, Variable>();

  for (const variable of Variables) {
    Var2Int.set(variable, count);
    Int2Var.set(count, variable);
    count++;
  }

  return [Var2Int, Int2Var];
}

The function `literal2int` takes a literal and transforms this literal into an integer
representing the literal.  If the literal is a negated variable, the integer is negative, else it is positive.
`Var2Int` is a dictionary mapping propositional variables to natural numbers.

In [14]:
function literal2int(literal: Literal, Var2Int: Map<Variable, number>): number {
  if (Array.isArray(literal) && literal[0] === '¬') {
    // Extract string variable name for lookup
    return -Var2Int.get(literal[1])!;
  } else if (typeof literal === 'string') {
    return Var2Int.get(literal)!;
  } else {
    throw new Error("Invalid literal format");
  }
}


The function `clause2pyco(Clause, Var2Int)` transforms a set of literals into a list of integers.
`Var2Int` is a dictionary mapping the propositional variables to integers.

In [15]:
function clause2pyco(Clause: Set<Literal>, Var2Int: Map<Variable, number>): number[] {
  return Array.from(Clause).map(literal => literal2int(literal, Var2Int));
}

The function `clauses2pyco(Clauses, Var2Int)` transforms a set of `Clauses` into a list of lists of integers.
`Var2Int` is a dictionary mapping the propositional variables to integers.

In [16]:
function clauses2pyco(Clauses: Set<Set<Literal>>, Var2Int: Map<Variable, number>): number[][] {
  return Array.from(Clauses).map(clause => clause2pyco(clause, Var2Int));
}

The function `int2var(Numbers, Int2Var)` takes a list of numbers representing a set of literals
and returns the associated list of literals.

In [17]:
function int2var(Numbers: number[], Int2Var: Map<number, Variable>): Set<Set<Literal>> {
  const Result = new Set<Set<Literal>>();
  for (const n of Numbers) {
    if (n > 0) {
      Result.add(new Set<Literal>([Int2Var.get(n)!]));
    } else {
      Result.add(new Set<Literal>([['¬', Int2Var.get(-n)!]]));
    }
  }
  return Result;
}

## Sudoku

The finish mathematician Arto Inkala claims to have created the [hardest sudoku](https://abcnews.go.com/blogs/headlines/2012/06/can-you-solve-the-hardest-ever-sudoku) ever.  It is defined below.

In [18]:
function createPuzzle(): (number | string)[][] {
  return [
    [ 8 , "*", "*", "*", "*", "*", "*", "*", "*"], 
    ["*", "*",  3 ,  6 , "*", "*", "*", "*", "*"],
    ["*",  7 , "*", "*",  9 , "*",  2 , "*", "*"],
    ["*",  5 , "*", "*", "*",  7 , "*", "*", "*"],
    ["*", "*", "*", "*",  4 ,  5 ,  7 , "*", "*"],
    ["*", "*", "*",  1 , "*", "*", "*",  3 , "*"],
    ["*", "*",  1 , "*", "*", "*", "*",  6 ,  8 ],
    ["*", "*",  8 ,  5 , "*", "*", "*",  1 , "*"],
    ["*",  9 , "*", "*", "*", "*",  4 , "*", "*"]
  ];
}

We will solve this Sudoku using the Davis-Putnam algorithm.  We use the following variables:
* `Q<r,c,d>` is a Boolean variable stating that the field in row `r` and column `c` holds the digit `d`.
  Here, `r`, `c`, `d` are all elements from the set $\{1,\cdots,9\}$.
    
The function `var(row, col, digit)` returns a formated string that is interpreted as a variable name.

In [19]:
function varName(row: number, col: number, digit: number): string {
    return `Q<${row},${col},${digit}>`;
}

In [20]:
varName(1,2,3);

Q<1,2,3>


The function `atMostOne(S)` takes a set `S` of propositional variables as its argument.  It returns a set of clauses
expressing the fact that at most one of the variables of `S` is true.

In [21]:
function atMostOne(S: Set<Variable>): Set<Clause> {
    const result = new Set<Clause>();
    const arr = Array.from(S);
    for (let i = 0; i < arr.length; i++) {
        for (let j = i + 1; j < arr.length; j++) {
            const p = arr[i];
            const q = arr[j];
            result.add(new Set<Literal>([['¬', p], ['¬', q]]));
        }
    }
    return result;
}

The function `atLeastOne(S)` takes a set `S` of propositional variables as its argument.  It returns a set of clauses
expressing the fact that at least one of the variables of `S` is true.

In [22]:
function atLeastOne(S: Set<Variable>): Set<Clause> {
    return new Set([new Set(S)]);
}

The function `exactlyOne(S)` takes a set `S` of propositional variables as its argument.  It returns a set of clauses
expressing the fact that exactly one of the variables of `S` is true.

In [23]:
function exactlyOne(S: Set<Variable>): Set<Clause> {
    const atMost = atMostOne(S);
    const atLeast = atLeastOne(S);
    return new Set([...atMost, ...atLeast]);
}

In [24]:
exactlyOne(new Set(['a', 'b', 'c']));

Set(4) {
  Set(2) { [ '¬', 'a' ], [ '¬', 'b' ] },
  Set(2) { [ '¬', 'a' ], [ '¬', 'c' ] },
  Set(2) { [ '¬', 'b' ], [ '¬', 'c' ] },
  Set(3) { 'a', 'b', 'c' }
}


The function `exactlyOnce` takes a list `L` of pairs of indices as its argument.  The elements of `L` are pairs of the form
`(row, col)`, where both `row` and `col` are elements of the set $\{1, \cdots, 9\}$.
It returns a set of formulas expressing that all Sudoku fields specified by the coordinate pairs in `L` take different digits as values.

In [25]:
function exactlyOnce(L: Array<[number, number]>): Set<Clause> {
    const Clauses = new Set<Clause>();
    for (let digit = 1; digit <= 9; digit++) {
        const vars = new Set<Variable>(
            L.map(([row, col]) => varName(row, col, digit))
        );
        const exact = exactlyOne(vars);
        for (const clause of exact) {
            Clauses.add(clause);
        }
    }
    return Clauses;
}

In [26]:
const result = exactlyOnce(
    Array.from({ length: 9 }, (_, i) => [1, i + 1] as [number, number])
);
console.log(result.size)
for (const clause of result) {
    console.log(`{${[...clause].map(lit => JSON.stringify(lit)).join(', ')}}`);
}

333
{["¬","Q<1,1,1>"], ["¬","Q<1,2,1>"]}
{["¬","Q<1,1,1>"], ["¬","Q<1,3,1>"]}
{["¬","Q<1,1,1>"], ["¬","Q<1,4,1>"]}
{["¬","Q<1,1,1>"], ["¬","Q<1,5,1>"]}
{["¬","Q<1,1,1>"], ["¬","Q<1,6,1>"]}
{["¬","Q<1,1,1>"], ["¬","Q<1,7,1>"]}
{["¬","Q<1,1,1>"], ["¬","Q<1,8,1>"]}
{["¬","Q<1,1,1>"], ["¬","Q<1,9,1>"]}
{["¬","Q<1,2,1>"], ["¬","Q<1,3,1>"]}
{["¬","Q<1,2,1>"], ["¬","Q<1,4,1>"]}
{["¬","Q<1,2,1>"], ["¬","Q<1,5,1>"]}
{["¬","Q<1,2,1>"], ["¬","Q<1,6,1>"]}
{["¬","Q<1,2,1>"], ["¬","Q<1,7,1>"]}
{["¬","Q<1,2,1>"], ["¬","Q<1,8,1>"]}
{["¬","Q<1,2,1>"], ["¬","Q<1,9,1>"]}
{["¬","Q<1,3,1>"], ["¬","Q<1,4,1>"]}
{["¬","Q<1,3,1>"], ["¬","Q<1,5,1>"]}
{["¬","Q<1,3,1>"], ["¬","Q<1,6,1>"]}
{["¬","Q<1,3,1>"], ["¬","Q<1,7,1>"]}
{["¬","Q<1,3,1>"], ["¬","Q<1,8,1>"]}
{["¬","Q<1,3,1>"], ["¬","Q<1,9,1>"]}
{["¬","Q<1,4,1>"], ["¬","Q<1,5,1>"]}
{["¬","Q<1,4,1>"], ["¬","Q<1,6,1>"]}
{["¬","Q<1,4,1>"], ["¬","Q<1,7,1>"]}
{["¬","Q<1,4,1>"], ["¬","Q<1,8,1>"]}
{["¬","Q<1,4,1>"], ["¬","Q<1,9,1>"]}
{["¬","Q<1,5,1>"], ["¬","Q<1,6,1>"

{["¬","Q<1,5,6>"], ["¬","Q<1,7,6>"]}
{["¬","Q<1,5,6>"], ["¬","Q<1,8,6>"]}
{["¬","Q<1,5,6>"], ["¬","Q<1,9,6>"]}
{["¬","Q<1,6,6>"], ["¬","Q<1,7,6>"]}
{["¬","Q<1,6,6>"], ["¬","Q<1,8,6>"]}
{["¬","Q<1,6,6>"], ["¬","Q<1,9,6>"]}
{["¬","Q<1,7,6>"], ["¬","Q<1,8,6>"]}
{["¬","Q<1,7,6>"], ["¬","Q<1,9,6>"]}
{["¬","Q<1,8,6>"], ["¬","Q<1,9,6>"]}
{"Q<1,1,6>", "Q<1,2,6>", "Q<1,3,6>", "Q<1,4,6>", "Q<1,5,6>", "Q<1,6,6>", "Q<1,7,6>", "Q<1,8,6>", "Q<1,9,6>"}
{["¬","Q<1,1,7>"], ["¬","Q<1,2,7>"]}
{["¬","Q<1,1,7>"], ["¬","Q<1,3,7>"]}
{["¬","Q<1,1,7>"], ["¬","Q<1,4,7>"]}
{["¬","Q<1,1,7>"], ["¬","Q<1,5,7>"]}
{["¬","Q<1,1,7>"], ["¬","Q<1,6,7>"]}
{["¬","Q<1,1,7>"], ["¬","Q<1,7,7>"]}
{["¬","Q<1,1,7>"], ["¬","Q<1,8,7>"]}
{["¬","Q<1,1,7>"], ["¬","Q<1,9,7>"]}
{["¬","Q<1,2,7>"], ["¬","Q<1,3,7>"]}
{["¬","Q<1,2,7>"], ["¬","Q<1,4,7>"]}
{["¬","Q<1,2,7>"], ["¬","Q<1,5,7>"]}
{["¬","Q<1,2,7>"], ["¬","Q<1,6,7>"]}
{["¬","Q<1,2,7>"], ["¬","Q<1,7,7>"]}
{["¬","Q<1,2,7>"], ["¬","Q<1,8,7>"]}
{["¬","Q<1,2,7>"], ["¬","Q<1,9,7>"]}
{["

The function `exactlyOneDigit(row, col)` takes integers `row` and `col` as arguments.  These specify the row and column of a field in a Sudoku.  The function returns a set of clauses specifying that exactly one of the variables

* `Q<row,col,1>`, `Q<row,col,2>`, $\cdots$, `Q<row,col,9>`

is `True`.

In [27]:
function exactlyOneDigit(row: number, col: number): Set<Clause> {
    const vars = new Set<Variable>();
    for (let digit = 1; digit <= 9; digit++) {
        vars.add(varName(row, col, digit));
    }
    return exactlyOne(vars);
}

In [28]:
exactlyOneDigit(1, 1);

Set(37) {
  Set(2) { [ '¬', 'Q<1,1,1>' ], [ '¬', 'Q<1,1,2>' ] },
  Set(2) { [ '¬', 'Q<1,1,1>' ], [ '¬', 'Q<1,1,3>' ] },
  Set(2) { [ '¬', 'Q<1,1,1>' ], [ '¬', 'Q<1,1,4>' ] },
  Set(2) { [ '¬', 'Q<1,1,1>' ], [ '¬', 'Q<1,1,5>' ] },
  Set(2) { [ '¬', 'Q<1,1,1>' ], [ '¬', 'Q<1,1,6>' ] },
  Set(2) { [ '¬', 'Q<1,1,1>' ], [ '¬', 'Q<1,1,7>' ] },
  Set(2) { [ '¬', 'Q<1,1,1>' ], [ '¬', 'Q<1,1,8>' ] },
  Set(2) { [ '¬', 'Q<1,1,1>' ], [ '¬', 'Q<1,1,9>' ] },
  Set(2) { [ '¬', 'Q<1,1,2>' ], [ '¬', 'Q<1,1,3>' ] },
  Set(2) { [ '¬', 'Q<1,1,2>' ], [ '¬', 'Q<1,1,4>' ] },
  Set(2) { [ '¬', 'Q<1,1,2>' ], [ '¬', 'Q<1,1,5>' ] },
  Set(2) { [ '¬', 'Q<1,1,2>' ], [ '¬', 'Q<1,1,6>' ] },
  Set(2) { [ '¬', 'Q<1,1,2>' ], [ '¬', 'Q<1,1,7>' ] },
  Set(2) { [ '¬', 'Q<1,1,2>' ], [ '¬', 'Q<1,1,8>' ] },
  Set(2) { [ '¬', 'Q<1,1,2>' ], [ '¬', 'Q<1,1,9>' ] },
  Set(2) { [ '¬', 'Q<1,1,3>' ], [ '¬', 'Q<1,1,4>' ] },
  Set(2) { [ '¬', 'Q<1,1,3>' ], [ '¬', 'Q<1,1,5>' ] },
  Set(2) { [ '¬', 'Q<1,1,3>' ], [ '¬', 'Q<1,1,6>' ] },


The function `constraints_from_puzzle`  returns a set of clauses stating that the variables corresponding to numbers that are already given in the Sudoku puzzle take the values that are specified.

In [29]:
function constraintsFromPuzzle(): Set<Clause> {
    const Puzzle = createPuzzle();
    const Variables: Variable[] = [];
    for (let row = 0; row < 9; row++) {
        for (let col = 0; col < 9; col++) {
            if (Puzzle[row][col] !== '*') {
                Variables.push(varName(row + 1, col + 1, Puzzle[row][col] as number));
            }
        }
    }
    return new Set(Variables.map(v => new Set<Literal>([v])));
}

In [30]:
constraintsFromPuzzle();

Set(21) {
  Set(1) { 'Q<1,1,8>' },
  Set(1) { 'Q<2,3,3>' },
  Set(1) { 'Q<2,4,6>' },
  Set(1) { 'Q<3,2,7>' },
  Set(1) { 'Q<3,5,9>' },
  Set(1) { 'Q<3,7,2>' },
  Set(1) { 'Q<4,2,5>' },
  Set(1) { 'Q<4,6,7>' },
  Set(1) { 'Q<5,5,4>' },
  Set(1) { 'Q<5,6,5>' },
  Set(1) { 'Q<5,7,7>' },
  Set(1) { 'Q<6,4,1>' },
  Set(1) { 'Q<6,8,3>' },
  Set(1) { 'Q<7,3,1>' },
  Set(1) { 'Q<7,8,6>' },
  Set(1) { 'Q<7,9,8>' },
  Set(1) { 'Q<8,3,8>' },
  Set(1) { 'Q<8,4,5>' },
  Set(1) { 'Q<8,8,1>' },
  Set(1) { 'Q<9,2,9>' },
  Set(1) { 'Q<9,7,4>' }
}


The function `all_constraints` returns a CSP that encodes the given sudoku as a CSP.

In [31]:
function literalKey(lit: Literal): string {
    if (typeof lit === 'string') return lit;
    return `¬${lit[1]}`;
}

function equalClauses(c1: Clause, c2: Clause): boolean {
    if (c1.size !== c2.size) return false;
    for (const lit of c1) {
        let found = false;
        for (const l2 of c2) {
            if (literalKey(l2) === literalKey(lit)) {
                found = true;
                break;
            }
        }
        if (!found) return false;
    }
    return true;
}

function addClauseUnique(Clauses: Set<Clause>, clause: Clause) {
    for (const c of Clauses) {
        if (equalClauses(c, clause)) return; // Clause already present
    }
    Clauses.add(clause);
}

function allConstraints(): Set<Clause> {
    const L = [1, 2, 3, 4, 5, 6, 7, 8, 9];
    let Clauses = constraintsFromPuzzle();

    // Exactly one digit in every cell
    for (const row of L) {
        for (const col of L) {
            for (const clause of exactlyOneDigit(row, col)) {
                addClauseUnique(Clauses, clause);
            }
        }
    }

    // Unique entries in each row
    for (const row of L) {
        for (const clause of exactlyOnce(L.map(col => [row, col] as [number, number]))) {
            addClauseUnique(Clauses, clause);
        }
    }

    // Unique entries in each column
    for (const col of L) {
        for (const clause of exactlyOnce(L.map(row => [row, col] as [number, number]))) {
            addClauseUnique(Clauses, clause);
        }
    }

    // Unique entries in each 3x3 square
    for (let r = 0; r < 3; r++) {
        for (let c = 0; c < 3; c++) {
            const blockCells: Array<[number, number]> = [];
            for (let row = 1; row <= 3; row++) {
                for (let col = 1; col <= 3; col++) {
                    blockCells.push([r * 3 + row, c * 3 + col]);
                }
            }
            for (const clause of exactlyOnce(blockCells)) {
                addClauseUnique(Clauses, clause);
            }
        }
    }

    return Clauses;
}

In [32]:
const clauses = Array.from(allConstraints());

for (const clause of clauses) {
    if (clause.size === 1) {
        console.log(clause);
    }
}

for (const clause of clauses) {
    if (clause.size === 9) {
        console.log(clause);
    }
}

Set(1) { 'Q<1,1,8>' }
Set(1) { 'Q<2,3,3>' }
Set(1) { 'Q<2,4,6>' }
Set(1) { 'Q<3,2,7>' }
Set(1) { 'Q<3,5,9>' }
Set(1) { 'Q<3,7,2>' }
Set(1) { 'Q<4,2,5>' }
Set(1) { 'Q<4,6,7>' }
Set(1) { 'Q<5,5,4>' }
Set(1) { 'Q<5,6,5>' }
Set(1) { 'Q<5,7,7>' }
Set(1) { 'Q<6,4,1>' }
Set(1) { 'Q<6,8,3>' }
Set(1) { 'Q<7,3,1>' }
Set(1) { 'Q<7,8,6>' }
Set(1) { 'Q<7,9,8>' }
Set(1) { 'Q<8,3,8>' }
Set(1) { 'Q<8,4,5>' }
Set(1) { 'Q<8,8,1>' }
Set(1) { 'Q<9,2,9>' }
Set(1) { 'Q<9,7,4>' }
Set(9) {
  'Q<1,1,1>',
  'Q<1,1,2>',
  'Q<1,1,3>',
  'Q<1,1,4>',
  'Q<1,1,5>',
  'Q<1,1,6>',
  'Q<1,1,7>',
  'Q<1,1,8>',
  'Q<1,1,9>'
}
Set(9) {
  'Q<1,2,1>',
  'Q<1,2,2>',
  'Q<1,2,3>',
  'Q<1,2,4>',
  'Q<1,2,5>',
  'Q<1,2,6>',
  'Q<1,2,7>',
  'Q<1,2,8>',
  'Q<1,2,9>'
}
Set(9) {
  'Q<1,3,1>',
  'Q<1,3,2>',
  'Q<1,3,3>',
  'Q<1,3,4>',
  'Q<1,3,5>',
  'Q<1,3,6>',
  'Q<1,3,7>',
  'Q<1,3,8>',
  'Q<1,3,9>'
}
Set(9) {
  'Q<1,4,1>',
  'Q<1,4,2>',
  'Q<1,4,3>',
  'Q<1,4,4>',
  'Q<1,4,5>',
  'Q<1,4,6>',
  'Q<1,4,7>',
  'Q<1,4,8>',
  'Q<1,4,

Set(9) {
  'Q<4,8,1>',
  'Q<4,8,2>',
  'Q<4,8,3>',
  'Q<4,8,4>',
  'Q<4,8,5>',
  'Q<4,8,6>',
  'Q<4,8,7>',
  'Q<4,8,8>',
  'Q<4,8,9>'
}
Set(9) {
  'Q<4,9,1>',
  'Q<4,9,2>',
  'Q<4,9,3>',
  'Q<4,9,4>',
  'Q<4,9,5>',
  'Q<4,9,6>',
  'Q<4,9,7>',
  'Q<4,9,8>',
  'Q<4,9,9>'
}
Set(9) {
  'Q<5,1,1>',
  'Q<5,1,2>',
  'Q<5,1,3>',
  'Q<5,1,4>',
  'Q<5,1,5>',
  'Q<5,1,6>',
  'Q<5,1,7>',
  'Q<5,1,8>',
  'Q<5,1,9>'
}
Set(9) {
  'Q<5,2,1>',
  'Q<5,2,2>',
  'Q<5,2,3>',
  'Q<5,2,4>',
  'Q<5,2,5>',
  'Q<5,2,6>',
  'Q<5,2,7>',
  'Q<5,2,8>',
  'Q<5,2,9>'
}
Set(9) {
  'Q<5,3,1>',
  'Q<5,3,2>',
  'Q<5,3,3>',
  'Q<5,3,4>',
  'Q<5,3,5>',
  'Q<5,3,6>',
  'Q<5,3,7>',
  'Q<5,3,8>',
  'Q<5,3,9>'
}
Set(9) {
  'Q<5,4,1>',
  'Q<5,4,2>',
  'Q<5,4,3>',
  'Q<5,4,4>',
  'Q<5,4,5>',
  'Q<5,4,6>',
  'Q<5,4,7>',
  'Q<5,4,8>',
  'Q<5,4,9>'
}
Set(9) {
  'Q<5,5,1>',
  'Q<5,5,2>',
  'Q<5,5,3>',
  'Q<5,5,4>',
  'Q<5,5,5>',
  'Q<5,5,6>',
  'Q<5,5,7>',
  'Q<5,5,8>',
  'Q<5,5,9>'
}
Set(9) {
  'Q<5,6,1>',
  'Q<5,6,2>',
  'Q<5,6,3>

In [ ]:
allConstraints().size

The function `solve(Constraints, Variables)` receives two arguments:
- `Constraints` is a set of formulas representing a constraint satisfaction problem.
- `Variables`   is the set of variables that occur in this formulas.

The function computes a solution to the given problem and returns this solution.

In [57]:
import {  Model } from "z3-solver";

async function sudoku(): Promise<Model<""> | null> {
  const Z3 = await init();
  const ctx = Z3.Context("");

  // Get constraints as Set of Clauses (Sets of Literals)
  const ClausesSet: Set<Set<Literal>> = allConstraints();

  // Convert clauses and literals to Z3 boolean expressions
  const clausesZ3 = Array.from(ClausesSet).map(clause => {
    const literalsZ3 = Array.from(clause).map(lit => {
      if (Array.isArray(lit) && lit[0] === '¬') {
        const v = ctx.Bool.const(lit[1]);
        return ctx.Not(v);
      } else if (typeof lit === 'string') {
        return ctx.Bool.const(lit);
      } else {
        throw new Error('Invalid literal format');
      }
    });
    return ctx.Or(...literalsZ3);
  });

  // Combine clauses with And (CNF)
  const formula = ctx.And(...clausesZ3);

  // Create solver and add formula
  const solver = new ctx.Solver();
  solver.add(formula);

  // Check satisfiability
  const result = await solver.check();

  if (result.toString() === "sat") {
    const model = solver.model();
    console.log("Model found.");
    return model;
  } else {
    console.log("The problem is not solvable!");
    return null;
  }
}

In [58]:
9**3

729


Even though this Sudoku is modelled using $9^3 = 729$ propositional variables and we have 10551 clauses, PicoSat uses less than 40 milliseconds to solve the problem. 

In [59]:
console.time("sudoku");
const Solution = await sudoku();
console.timeEnd("sudoku");

Model found.
sudoku: 13.739s


In [75]:
Solution.toString()

(define-fun |Q<5,6,5>| () Bool
  true)
(define-fun |Q<4,7,4>| () Bool
  false)
(define-fun |Q<1,6,6>| () Bool
  false)
(define-fun |Q<2,8,4>| () Bool
  false)
(define-fun |Q<3,6,9>| () Bool
  false)
(define-fun |Q<8,6,3>| () Bool
  false)
(define-fun |Q<6,2,3>| () Bool
  false)
(define-fun |Q<1,5,1>| () Bool
  false)
(define-fun |Q<5,6,3>| () Bool
  false)
(define-fun |Q<7,5,7>| () Bool
  true)
(define-fun |Q<2,1,8>| () Bool
  false)
(define-fun |Q<8,7,6>| () Bool
  false)
(define-fun |Q<5,4,9>| () Bool
  false)
(define-fun |Q<7,9,3>| () Bool
  false)
(define-fun |Q<8,9,2>| () Bool
  false)
(define-fun |Q<8,2,1>| () Bool
  false)
(define-fun |Q<6,5,1>| () Bool
  false)
(define-fun |Q<5,7,5>| () Bool
  false)
(define-fun |Q<2,9,2>| () Bool
  false)
(define-fun |Q<1,4,6>| () Bool
  false)
(define-fun |Q<4,8,4>| () Bool
  false)
(define-fun |Q<6,9,3>| () Bool
  false)
(define-fun |Q<3,7,5>| () Bool
  false)
(define-fun |Q<6,9,7>| () Bool
  false)
(define-fun |Q<1,4,7>| () Bool
  true)
(de

The function `remove_negative` removes the negative literals from the given solution `S` and returns the set of variables that have to be `True`.  

In [77]:
function remove_negative_from_model_string(modelStr: string): Set<string> {
  const literals = new Set<string>();
  const regex = /\(define-fun \|([^|]+)\| \(\) Bool\s+(true|false)\)/g;
  let match;

  while ((match = regex.exec(modelStr)) !== null) {
    const varName = match[1];  // variable like Q<5,6,5>
    const value = match[2];    // "true" or "false"

    if (value === "true") {
      literals.add(varName);   // Add only positive literals (true)
    }
    // Skip literals with value false (negated)
  }

  return literals;
}

In [78]:
const SolutionNoNegative = remove_negative_from_model_string(Solution.toString());

In [79]:
SolutionNoNegative

Set(81) {
  'Q<5,6,5>',
  'Q<7,5,7>',
  'Q<1,4,7>',
  'Q<9,1,7>',
  'Q<5,9,1>',
  'Q<5,4,8>',
  'Q<1,6,3>',
  'Q<4,1,1>',
  'Q<5,8,2>',
  'Q<3,5,9>',
  'Q<1,8,4>',
  'Q<3,9,3>',
  'Q<3,3,5>',
  'Q<6,6,9>',
  'Q<2,3,3>',
  'Q<4,9,6>',
  'Q<9,9,2>',
  'Q<1,1,8>',
  'Q<4,8,9>',
  'Q<6,5,6>',
  'Q<8,6,6>',
  'Q<7,7,3>',
  'Q<6,7,5>',
  'Q<6,1,2>',
  'Q<3,7,2>',
  'Q<2,1,9>',
  'Q<3,8,8>',
  'Q<3,2,7>',
  'Q<7,1,5>',
  'Q<3,4,4>',
  'Q<4,3,4>',
  'Q<6,2,8>',
  'Q<5,3,9>',
  'Q<1,7,6>',
  'Q<8,2,3>',
  'Q<7,2,2>',
  'Q<1,9,9>',
  'Q<2,2,4>',
  'Q<5,1,3>',
  'Q<3,1,6>',
  'Q<3,6,1>',
  'Q<7,6,4>',
  'Q<1,3,2>',
  'Q<8,1,4>',
  'Q<2,5,8>',
  'Q<2,4,6>',
  'Q<2,9,5>',
  'Q<7,9,8>',
  'Q<4,2,5>',
  'Q<5,5,4>',
  'Q<9,8,5>',
  'Q<6,4,1>',
  'Q<1,5,5>',
  'Q<8,5,2>',
  'Q<6,9,4>',
  'Q<6,3,7>',
  'Q<4,4,2>',
  'Q<9,7,4>',
  'Q<7,8,6>',
  'Q<4,5,3>',
  'Q<4,7,8>',
  'Q<8,9,7>',
  'Q<8,3,8>',
  'Q<4,6,7>',
  'Q<9,4,3>',
  'Q<2,6,2>',
  'Q<1,2,1>',
  'Q<2,7,1>',
  'Q<5,2,6>',
  'Q<9,6,8>',
  'Q<9,5,1

## Graphical Representation

The following line needs to be executed once to install the package `problem_visuals`.

In [80]:
function showSolutionFromSet(Solution: Set<string>, width: string = '50%'): void {
  // Create initial puzzle grid
  const Sudoku = createPuzzle();

  // Map to hold solution values keyed by "Q<row,col,digit>" strings
  const solutionMap: Record<string, number> = {};

  // Parse solution literals of form Q<r,c,d>, set digit d at (r,c)
  for (const literal of Solution) {
    // Extract row, col, digit from literal string like 'Q<5,6,5>'
    const match = literal.match(/^Q<(\d),(\d),(\d)>$/);
    if (match) {
      const row = parseInt(match[1], 10) - 1;
      const col = parseInt(match[2], 10) - 1;
      const digit = parseInt(match[3], 10);
      solutionMap[`V${row + 1}${col + 1}`] = digit;
    }
  }

  // Remove fixed cells from solutionMap (cells already given in puzzle)
  for (let row = 0; row < 9; row++) {
    for (let col = 0; col < 9; col++) {
      if (Sudoku[row][col] !== '*') {
        delete solutionMap[`V${row + 1}${col + 1}`];
      }
    }
  }

  // Build HTML table string with appropriate cell values
  let html = `<table style="width:${width}; border-collapse: collapse; border: 2px solid black;">`;
  for (let row = 0; row < 9; row++) {
    html += '<tr>';
    for (let col = 0; col < 9; col++) {
      const key = `V${row + 1}${col + 1}`;
      const value = solutionMap[key] || Sudoku[row][col];
      // Alternate box coloring for 3x3 blocks
      const boxColor = Math.floor(row / 3) % 2 === Math.floor(col / 3) % 2 ? '#f0f0f0' : '#ffffff';
      html += `<td style="border:1px solid #888; width:30px; height:30px; text-align:center; font-size:16px; background-color:${boxColor};">${value}</td>`;
    }
    html += '</tr>';
  }
  html += '</table>';

  // Display HTML in tslab environment
  tslab.display.html(html);
}

In [82]:
showSolutionFromSet(SolutionNoNegative);

8,1,2,7,5,3,6,4,9
9,4,3,6,8,2,1,7,5
6,7,5,4,9,1,2,8,3
1,5,4,2,3,7,8,9,6
3,6,9,8,4,5,7,2,1
2,8,7,1,6,9,5,3,4
5,2,1,9,7,4,3,6,8
4,3,8,5,2,6,9,1,7
7,9,6,3,1,8,4,5,2
